In [4]:
# Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [5]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
# Report Builder
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html

In [7]:
# PromptEvaluator Implementation
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        prompt = """
        Genera {num_cases} ideas únicas y diversas para probar un prompt que cumple con esta tarea:
        
        <task_description>
        {task_description}
        </task_description>

        El prompt recibirá las siguientes entradas:
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Cada idea debe representar un escenario o ejemplo distinto que ponga a prueba diferentes aspectos de la tarea.
        
        Formato de salida:
        Proporciona tu respuesta como una matriz JSON estructurada donde cada elemento sea una breve descripción de la idea.
        
        Ejemplo:
        ```json
        [
            "Prueba con terminología técnica de informática",
            "Prueba con hallazgos de investigación médica",
            "Prueba con conceptos matemáticos complejos",
            ...
        ]
        ```
        
        Asegúrate de que cada idea sea:
        - Claramente distinta de las demás
        - Relevante para la descripción de la tarea
        - Lo suficientemente específica como para guiar la generación de un caso de prueba completo
        - Rápida de resolver sin requerir un cálculo extensivo ni procesamiento de múltiples pasos
        - Resoluble con no más de 400 tokens de salida

        Recuerda, genera únicamente {num_cases} ideas únicas
        """

        system_prompt = "Eres un diseñador de escenarios de prueba especializado en crear escenarios de prueba diversos y únicos."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generar un único caso de prueba basado en la descripción de la tarea y una idea específica"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Genera un único caso de prueba detallado para la evaluación de un prompt basado en:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Formato de salida:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterio 1", "criterio 2", ...] // Lista concisa de criterios para evaluar la solución, de 1 a 4 elementos
        }}
        ```
        
        REQUISITOS IMPORTANTES:
        - DEBES USAR EXCLUSIVAMENTE estas claves de entrada exactas en tus prompt_inputs: {allowed_keys}        
        - NO agregues ninguna clave adicional a prompt_inputs
        - Todas las claves listadas en allowed_input_keys deben estar incluidas en tu respuesta
        - Haz que el caso de prueba sea realista y útil en la práctica
        - Incluye criterios de solución medibles y concisos
        - Los criterios de solución DEBEN abordar únicamente los requisitos directos de la descripción de la tarea y de los prompt_inputs generados
        - Evita sobreespecificar criterios con requisitos que vayan más allá de la tarea principal
        - Mantén los criterios de solución simples, enfocados y directamente vinculados a la tarea fundamental
        - El caso de prueba debe adaptarse a la idea específica proporcionada
        - Rápido de resolver sin requerir un cálculo extensivo ni procesamiento de múltiples pasos
        - Resoluble con no más de 400 tokens de salida
        - NO incluyas ningún campo más allá de los especificados en el formato de salida

        Aquí tienes un ejemplo de entrada de muestra con una salida ideal:
        <sample_input>
        <sample_task_description>
        Extraer temas de un fragmento de texto
        </sample_task_description>
        <sample_specific_idea>
        Prueba con un texto que contenga múltiples temas y subtemas anidados (por ejemplo, un fragmento sobre energía renovable que abarque simultáneamente la economía de la energía solar, la tecnología de turbinas eólicas y las implicaciones políticas)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "La transición hacia la energía renovable abarca numerosas dimensiones interdependientes. La tecnología solar fotovoltaica ha experimentado reducciones drásticas de costos, con una eficiencia de los paneles que mejoró un 24% desde 2010, mientras que los costos de fabricación disminuyeron un 89%, lo que la hace económicamente competitiva frente a los combustibles fósiles en muchos mercados. Simultáneamente, la energía eólica ha evolucionado mediante diseños innovadores de turbinas con palas de compuesto de fibra de carbono y sistemas de control avanzados que aumentan la captura de energía en un 35% en condiciones de poco viento."
            },
            "solution_criteria": [
                "Incluye todos los temas mencionados"   
            ]
        }
        ```
        </ideal_output>
        Esta es una salida ideal porque los criterios de solución son concisos y no solicitan nada que esté fuera del alcance de la descripción de la tarea.
        """

        system_prompt = "Eres un creador de casos de prueba especializado en diseñar escenarios de evaluación."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Requisitos obligatorios - CUALQUIER VIOLACIÓN SIGNIFICA FALLO AUTOMÁTICO (puntuación de 3 o menos):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Tu tarea es evaluar la siguiente solución generada por IA con EXTREMO RIGOR.

        Descripción original de la tarea:
        <task_description>
        {task_description}
        </task_description>

        Entradas originales de la tarea:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solución a evaluar:
        <solution>
        {output}
        </solution>

        Criterios que debes utilizar para evaluar la solución:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Pautas de puntuación:
        * Puntuación 1-3: La solución no cumple con uno o más requisitos OBLIGATORIOS.
        * Puntuación 4-6: La solución cumple con todos los requisitos obligatorios, pero presenta deficiencias significativas en los criterios secundarios.
        * Puntuación 7-8: La solución cumple con todos los requisitos obligatorios y la mayoría de los criterios secundarios, con problemas menores.
        * Puntuación 9-10: La solución cumple con todos los criterios obligatorios y secundarios.

        INSTRUCCIONES IMPORTANTES DE PUNTUACIÓN:
        * Califica la salida basándote ÚNICAMENTE en los criterios listados. No añadas tus propios requisitos adicionales.
        * Si una solución cumple con todos los criterios obligatorios y secundarios, asígnale un 10.
        * No te quejes de que la solución "solo" cumple con los requisitos obligatorios y secundarios. Las soluciones no deben ir más allá de lo esperado; deben cumplir exactamente con los criterios listados.
        * CUALQUIER violación de un requisito obligatorio DEBE resultar en una puntuación de 3 o menos.
        * Se debe utilizar toda la escala del 1 al 10; no dudes en dar puntuaciones bajas cuando esté justificado.

        Formato de salida:
        Proporciona tu evaluación como un objeto JSON estructurado con los siguientes campos, en este orden específico:
        - "strengths": Una matriz de 1 a 3 fortalezas clave
        - "weaknesses": Una matriz de 1 a 3 áreas clave de mejora
        - "reasoning": Una explicación concisa de tu evaluación general
        - "score": Un número entre el 1 y el 10

        Responde con un JSON. Mantén tu respuesta concisa y directa.
        Estructura de ejemplo para la respuesta:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [8]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [9]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Escribe un plan conciso y compacto de alimentación para un simple atleta.",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Altura del atleta en cm",
        "weight": "Peso del atleta en kg",
        "goal": "Objetivo del atleta",
        "restrictions": "Restricciones dietéticas del atleta",
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [21]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    Elabora un plan de alimentación de un día para un atleta que cumpla con sus restricciones dietéticas.

    <athlete_information>
    - Altura: {prompt_inputs["height"]}
    - Peso: {prompt_inputs["weight"]}
    - Objetivo: {prompt_inputs["goal"]}
    - Restricción dietética: {prompt_inputs["restrictions"]}
    </athlete_information>

    Directrices:
    1. Incluir la cantidad diaria exacta de calorías
    2. Indicar las cantidades de proteínas, grasas y carbohidratos
    3. Especificar el momento de cada comida
    4. Utilizar únicamente alimentos que cumplan con las restricciones
    5. Indicar el tamaño de todas las porciones en gramos
    6. Ajustarse al presupuesto, si se ha mencionado

    Aquí hay un ejemlo de entrada y una salida ideal para guiarte:
    <sample_input>
    height: 178
    weight: 75
    goal: Mejorar rendimiento explosivo y resistencia en futbol con presupuesto limitado
    restrictions: Sin alergias, presupuesto máximo de $200 USD mensuales
    </sample_input>
    <ideal_output>
    # Plan de Alimentación Diaria - Atleta Futbolista
    ## Perfil: 178 cm | 75 kg | Presupuesto Limitado

    ---

    ## 📊 CÁLCULO DE REQUERIMIENTOS

    **IMC:** 23.7 (Normal)  
    **Calorías diarias:** 2,800-3,000 kcal  
    **Distribución macros:**
    - Proteína: 120-130g (40%)
    - Carbohidratos: 350-375g (45%)
    - Grasas: 85-100g (30%)

    ---

    ## 🍽️ PLAN DE ALIMENTACIÓN (DÍA TIPO)

    ### **DESAYUNO** (7:00 AM) | 650 kcal
    ```
    - 3 huevos revueltos
    - 2 tazas de arroz blanco cocido
    - 1 plátano mediano
    - Café con leche descremada
    ```
    **Macro:** 24g proteína | 85g carbs | 18g grasa

    ---

    ### **ALMUERZO (Pre-entreno)** (11:30 AM) | 500 kcal
    ```
    - 150g de pechuga de pollo (hervida/asada)
    - 1.5 tazas de arroz blanco
    - Ensalada: lechuga, tomate
    - 1 cucharadita de aceite
    ```
    **Macro:** 40g proteína | 60g carbs | 8g grasa

    ---

    ### **PRE-ENTRENO** (1 hora antes del partido/entreno) | 250 kcal
    ```
    - 1-2 plátanos
    - 30g de avena (en agua o leche)
    - 1 taza de jugo natural de naranja
    ```
    **Macro:** 8g proteína | 55g carbs | 2g grasa

    ---

    ### **POST-ENTRENO** (30-45 min después) | 350 kcal
    ```
    - Licuado:
    - 1 taza de leche descremada
    - 2 huevos cocidos
    - 1 plátano
    - 2 cucharadas de avena
    - 1 cucharadita de miel
    ```
    **Macro:** 18g proteína | 50g carbs | 8g grasa

    ---

    ### **CENA** (7:00 PM) | 650 kcal
    ```
    - 180g de carne molida (7% grasa) o atún en lata
    - 2 tazas de arroz blanco
    - Frijoles/lentejas (1 taza cocida)
    - Ensalada: repollo, zanahoria rallada
    - 1 cucharadita de aceite
    ```
    **Macro:** 35g proteína | 85g carbs | 12g grasa

    ---

    ### **REFRIGERIO NOCTURNO** (Opcional - 9:30 PM) | 150 kcal
    ```
    - 200ml de leche descremada
    - 30g de pan integral o tortilla
    ```
    **Macro:** 8g proteína | 20g carbs | 2g grasa

    ---

    ## 📈 TOTALES DIARIOS
    | Nutriente | Cantidad | Objetivo |
    |-----------|----------|----------|
    | **Calorías** | 3,150 kcal | ✅ |
    | **Proteína** | 133g | ✅ |
    | **Carbohidratos** | 355g | ✅ |
    | **Grasas** | 50g | ✅ |

    ---

    ## 💰 PRESUPUESTO MENSUAL ESTIMADO

    | Alimento | Costo/mes |
    |----------|-----------|
    | Huevos (120 unid.) | $18 |
    | Arroz (15 kg) | $15 |
    | Pollo (9 kg) | $60 |  
    </ideal_output>
    Este es un ejemplo del razonamiento: La solución cumple de manera sólida con todos los requisitos obligatorios: proporciona total calórico diario claro (3,150 kcal), desglose detallado de macronutrientes con porcentajes y distribución por comida, y especifica alimentos con porciones exactas y horarios precisos. La distribución macronutriente es apropiada para fútbol (deporte intermitente) con énfasis en carbohidratos. El formato es conciso y práctico. Sin embargo, presenta deficiencias en criterios secundarios: el análisis de presupuesto es incompleto (solo 46% del presupuesto estimado), no menciona explícitamente micronutrientes clave aunque estén presentes en los alimentos, y carece de flexibilidad para diferentes tipos de días de entrenamiento. Estos problemas son significativos pero no violaciones de requisitos obligatorios.
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

In [22]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json", 
    extra_criteria="""
    La salida debería incluir:
    - Total calórico diario
    - Desglose de macronutrientes
    - Comidas con alimentos, porciones y horarios exactos
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 6.666666666666667
